In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import random

%matplotlib inline

In [ ]:
# Load the data and create a vocabulary of characters and words
root = Path('..')
data_path = root / 'data' / 'names.txt'
words = open(data_path).read().splitlines()
print(f'Words count: {len(words)}')

chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}
VOCAB_SIZE = len(stoi)
print(f'Vocabulary size: {VOCAB_SIZE}')
print(itos)

random.seed(42)
random.shuffle(words)

In [ ]:
# build dataset split into train / val / test
BLOCK_SIZE = 8 # The size of the context window

def build_dataset(words, block_size=3):
    X, Y = [], []

    for w in words:

        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, X.dtype, Y.shape, Y.dtype)
    return X, Y

X, Y = build_dataset(words, block_size=BLOCK_SIZE)

s1 = int(0.8 * len(X))
s2 = int(0.9 * len(X))

X_train, Y_train = X[:s1], Y[:s1]
X_val, Y_val = X[s1:s2], Y[s1:s2]
X_test, Y_test = X[s2:], Y[s2:]

print(f'train: {len(X_train)} val: {len(X_val)} test: {len(X_test)}')

In [ ]:
for x, y in zip(X_train[:30], Y_train[:30]):
    print(''.join(itos[ix.item()] for ix in x), '-->', itos[y.item()])

In [ ]:
class Linear:

    def __init__(self, fan_in, fan_out, bias=True):
        self.weight = torch.randn((fan_in, fan_out)) # / fan_in ** 0.5
        self.bias = torch.zeros(fan_out) if bias else None

    def __call__(self, x):
        self.out = x @ self.weight
        if self.bias is not None: self.out += self.bias
        return self.out
    
    def parameters(self):
        return [self.weight] + ([] if self.bias is None else [self.bias])
    
class BatchNorm1d:

    def __init__(self, dim, eps=1e-5, momentum=0.1):
        self.eps = eps
        self.momentum = momentum
        self.training = True

        # Parameters (trainable)
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)

        # Buffers (trained with a running 'momentum update')
        self.running_mean = torch.zeros(dim)
        self.running_var = torch.ones(dim)

    def __call__(self, x):

        # Forward pass
        if self.training:
            if x.ndim == 2:
                dim = 0
            elif x.ndim == 3:
                dim = (0, 1)
            xmean = x.mean(dim, keepdim=True) # batch mean
            xvar = x.var(dim, unbiased=True, keepdim=True) # batch variance
        else:
            xmean = self.running_mean
            xvar = self.running_var
        
        xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize
        self.out = self.gamma * xhat + self.beta

        # Update the buffers (inference mode)
        if self.training:
            with torch.no_grad():
                self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * xmean
                self.running_var = (1 - self.momentum) * self.running_var + self.momentum * xvar
        
        return self.out

    def parameters(self):
        return [self.gamma, self.beta]

class Embedding:

    def __init__(self, num_embeddings, embedding_dim):
        self.weight = torch.randn((num_embeddings, embedding_dim))
    
    def __call__(self, IX):
        self.out = self.weight[IX]
        return self.out

    def parameters(self):
        return [self.weight]

class FlattenConsecutive:
    
    def __init__(self, n):
        self.n = n

    def __call__(self, x):
        B, T, C = x.shape
        x = x.view(B, T // self.n, C * self.n)
        if x.shape[1] == 1:
            x = x.squeeze(1)
        self.out = x
        return self.out

    def parameters(self):
        return []

class Sequential:

    def __init__(self, layers):
        self.layers = layers
    
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        self.out = x
        return self.out

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

class Tanh:

    def __call__(self, x):
        self.out = torch.tanh(x)
        return self.out

    def parameters(self):
        return []

class ReLU:
    
    def __call__(self, x):
        self.out = x.clamp_min(0)
        return self.out

    def parameters(self):
        return []

In [ ]:
torch.manual_seed(42)

In [ ]:
N_EMBD = 24 # The dimensionality of the chars embeddings
N_HIDDEN = 200 # The number of hidden layers

model = Sequential([
    Embedding(VOCAB_SIZE, N_EMBD),
    FlattenConsecutive(2), Linear(N_EMBD * 2, N_HIDDEN, bias=False), BatchNorm1d(N_HIDDEN), Tanh(),
    FlattenConsecutive(2), Linear(N_HIDDEN * 2, N_HIDDEN, bias=False), BatchNorm1d(N_HIDDEN), Tanh(),
    FlattenConsecutive(2), Linear(N_HIDDEN * 2, N_HIDDEN, bias=False), BatchNorm1d(N_HIDDEN), Tanh(),
    Linear(N_HIDDEN, VOCAB_SIZE)
])

# parameter init
with torch.no_grad():
    model.layers[-1].weight *= 0.1

parameters = model.parameters()
print(f'Number of params: {sum(p.nelement() for p in parameters)}')
for p in parameters: p.requires_grad = True

In [ ]:
ix = torch.randint(0, X_train.shape[0], (4, )) # Random batch of 4 examples
Xb, Yb = X_train[ix], Y_train[ix]
logits = model(Xb)

for layer in model.layers:
    print(layer.__class__.__name__, layer.out.shape)

In [ ]:
EPOCHS = 100_000
BATCH_SIZE = 64
LOSSI = []
UD = [] # update to data ratio

for epoch in range(EPOCHS):

    # Minibatch
    ix = torch.randint(0, X_train.shape[0], (BATCH_SIZE,))
    Xb, Yb = X_train[ix], Y_train[ix]

    # Forward pass
    logits = model(Xb)
    loss = F.cross_entropy(logits, Yb)

    # Backward pass
    for p in parameters: p.grad = None
    loss.backward()

    # Update 
    lr = 0.1 if epoch < 150_000 else 0.01
    for p in parameters: p.data -= lr * p.grad

    # Track stats
    if epoch % 5_000 == 0: print(f'epoch: {epoch}/{EPOCHS} loss: {loss.item():.3f}')
    LOSSI.append(loss.log10().item())
    with torch.no_grad():
        UD.append([(lr * p.grad.std() / p.data.std()).log10().item() for p in parameters])

In [ ]:
meani = torch.tensor(LOSSI).view(-1, 2000).mean(1) # -1 in view means infer the size from the other dimensions
plt.plot(meani)

In [ ]:
# Eval model
for layer in model.layers:
    layer.training = False

In [ ]:
# Evaluation
@torch.no_grad()
def split_loss(split):
    x, y = {
        'train': (X_train, Y_train),
        'val': (X_val, Y_val),
        'test': (X_test, Y_test)
    }[split]
    logits = model(x)
    loss = F.cross_entropy(logits, y)
    print(f'{split} loss: {loss.item():.3f}')

split_loss('train')
split_loss('val')
split_loss('test')

In [ ]:
# Sample from the model

for _ in range(20):
    out = []
    context = [0] * BLOCK_SIZE
    while True:
        logits = model(torch.tensor([context]))
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1).item() # sample from normal distribution
        context = context[1:] + [ix] # shift the context window
        out.append(ix) # store the generated charachter
        if ix == 0: break # break if we sample the special character '.'
    print(''.join(itos[i] for i in out))